# 02 — Beverage Filtering

This notebook calls `src.beverage_filtering.get_beverage_products()` and shows how
the 4,910 beverage SKUs were identified from the 47K+ product catalog.  An allowlist
of `commodity_desc` strings was used; dairy milk and alcoholic beverages were
explicitly excluded to keep the scope on non-alcoholic packaged beverages relevant
to a Coca-Cola-style RGM analysis.

In [ ]:
import sys
import os
sys.path.insert(0, os.path.abspath('..'))
import pandas as pd
import warnings
warnings.filterwarnings('ignore')
print('Ready')

In [ ]:
from src.beverage_filtering import get_beverage_products
from src.data_cleaning import PROCESSED_DIR

bev = pd.read_parquet(PROCESSED_DIR / 'beverage_products.parquet')
print(f'Total beverage SKUs: {len(bev):,}')
print()
print('SKU count by beverage_category:')
print(bev['beverage_category'].value_counts().to_string())

In [ ]:
# Commodity-level breakdown within each category
commodity_counts = (
    bev.groupby(['beverage_category', 'commodity_desc'])
    .size()
    .reset_index(name='sku_count')
    .sort_values(['beverage_category', 'sku_count'], ascending=[True, False])
)
print(commodity_counts.to_string(index=False))

In [ ]:
# Revenue by category from beverage_transactions
bt = pd.read_parquet(PROCESSED_DIR / 'beverage_transactions.parquet')
rev_by_cat = bt.groupby('beverage_category')['sales_value'].sum().sort_values(ascending=False)
rev_pct = (rev_by_cat / rev_by_cat.sum() * 100).round(1)
print('Revenue share by category:')
for cat, pct in rev_pct.items():
    print(f'  {cat:<30} {pct:.1f}%')

In [ ]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import pathlib

CHARTS = pathlib.Path('..') / 'outputs' / 'charts'

sku_counts = bev['beverage_category'].value_counts().sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(9, 5))
ax.bar(sku_counts.index, sku_counts.values, color='#1f77b4', edgecolor='white')
for i, (cat, val) in enumerate(sku_counts.items()):
    ax.text(i, val + 15, str(val), ha='center', va='bottom', fontsize=9)
ax.set_title('Beverage SKU Count by Category', fontsize=12)
ax.set_xlabel('Beverage Category', fontsize=10)
ax.set_ylabel('Number of SKUs', fontsize=10)
ax.set_xticks(range(len(sku_counts)))
ax.set_xticklabels(sku_counts.index, rotation=20, ha='right', fontsize=9)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
plt.tight_layout()
out = CHARTS / 'beverage_sku_by_category.png'
fig.savefig(out, dpi=130)
plt.close(fig)
print(f'Chart saved: {out}')

## Summary — Beverage Filtering

- **4,910 beverage SKUs** identified from the full product catalog using a `commodity_desc` allowlist.
- **Carbonated Soft Drinks** is the largest category: 2,139 SKUs, **59.2% of beverage revenue** ($391K of $661K).
- Juice: 1,158 SKUs; Coffee: 604; Tea: 394; Other Beverages: 301; Energy/Sports Drinks: 240; Water: 74.
- Dairy milk excluded (outside non-alcoholic packaged beverages scope).
- Alcoholic beverages excluded (different regulatory/pricing dynamics).
- Total beverage revenue: **$661,333** — **8.2% of total store revenue** ($8,057,463).